# Agent 2: Retriever & Evaluator

## Τι κάνει αυτός ο agent;

Ο **Retriever & Evaluator** είναι ο πυρήνας του agentic σχεδιασμού. Εκτελεί το **ReAct loop**:

```
┌─────────────────────────────────────────────────────┐
│                   ReAct Loop                         │
│                                                     │
│  query ── RETRIEVE (ChromaDB) ── chunks           │
│                                        │            │
│               ── EVALUATE (LLM) ────┘            │
│               │                                     │
│         score ≥ 3?                                  │
│         ┌────┴────┐                                 │
│         YES       NO (& retries < 3)                │
│         │         │                                 │
│                                                   │
│       DONE    REFORMULATE query ── loop back       │
└─────────────────────────────────────────────────────┘
```

**Reason**  **Act**  **Observe**  **Decide**

Αυτός ο κύκλος είναι αυτό που κάνει τον agent *πραγματικά agentic*: δεν κάνει απλώς passthrough, αλλά **αξιολογεί** αν το αποτέλεσμα είναι αρκετά καλό και **αποφασίζει** αν πρέπει να ξαναπροσπαθήσει.

## Prerequisites

Αυτό το notebook **χρειάζεται το ChromaDB από το RAG_demo**.
Αν δεν έχετε τρέξει το `RAG_demo/rag_pipeline.ipynb`, κάντε το πρώτα.


In [1]:
from pathlib import Path
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from openai import OpenAI
import json

CHROMA_PATH = Path("../RAG_demo/chroma_db")
COLLECTION  = "arxiv_papers"

embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client       = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection   = client.get_collection(name=COLLECTION, embedding_function=embedding_fn)

print(f"Collection '{COLLECTION}' loaded: {collection.count()} chunks")

OLLAMA_BASE  = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2:3b"

llm = OpenAI(base_url=OLLAMA_BASE, api_key="ollama")

def call_llm(system_prompt: str, user_message: str) -> str:
    response = llm.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content

def extract_json(text: str) -> dict:
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError(f"No JSON in: {text!r}")
    return json.loads(text[start:end])

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 26002.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection 'arxiv_papers' loaded: 18654 chunks


## Βήμα 1: Retrieve

Ακριβώς όπως στο RAG demo — query embedding + cosine similarity search.

In [2]:
TOP_K = 5  # πόσα chunks να φέρνουμε ανά query

def retrieve(query: str, k: int = TOP_K) -> list[dict]:
    """Κάνει vector search και επιστρέφει τα top-k chunks με metadata."""
    results = collection.query(query_texts=[query], n_results=k)
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text":     doc,
            "title":    meta["title"],
            "arxiv_id": meta["arxiv_id"],
            "authors":  meta["authors"],
            "score":    round(1 - dist, 4),  # cosine similarity (higher = better)
        })
    return chunks

# Δοκιμή
test_chunks = retrieve("spiking neural network architecture")
print(f"Retrieved {len(test_chunks)} chunks:\n")
for i, c in enumerate(test_chunks, 1):
    print(f"[{i}] score={c['score']}  {c['title'][:60]}...")
    print(f"     {c['text'][:100]}...\n")

Retrieved 5 chunks:

[1] score=0.5408  LayerBoost: Layer-Aware Attention Reduction for Efficient LL...
     Koh, Ali Farhadi, Noah A. Smith, and Hannaneh Hajishirzi. Olmo 3, 2025. URL https:
//arxiv.org/abs/2...

[2] score=0.4592  Graph Memory Transformer (GMT)...
     if some internal objects are stable, named by their role in the architecture,
and repeatedly reused ...

[3] score=0.4507  Preference Heads in Large Language Models: A Mechanistic Fra...
     Processing Systems 2024, NeurIPS 2024, Vancouver,
BC, Canada, December 10 - 15, 2024.
Aryo Pradipta ...

[4] score=0.4443  Hyperloop Transformers...
     as a “depth-wise RNN” with matrix-valued hidden states Y(0), acts as the input at each
time (i.e., l...

[5] score=0.4399  Graph Memory Transformer (GMT)...
     matrices.
The architecture proposed here makes this memory-like computation more
structurally interp...



## Βήμα 2: Evaluate

Ο evaluator LLM αξιολογεί αν τα chunks **απαντούν πραγματικά** στο query.

Επιστρέφει:
- `score` (1–5): πόσο καλά απαντιέται το query
- `sufficient` (bool): αν είναι αρκετό για να γεννηθεί απάντηση
- `reformulated_query`: νέο query αν score < 3

> Αυτή είναι η **κρίσιμη διαφορά** από ένα απλό RAG: δεν εμπιστευόμαστε τυφλά τα retrieved chunks.

In [4]:
EVALUATOR_PROMPT = """
You are a Relevance Evaluator for an academic literature assistant.

You receive a search query and retrieved text passages from academic papers.
Assess whether the passages collectively contain enough information to answer the query.

Respond with ONLY valid JSON — no other text:
{
  "score": <integer 1-5>,
  "sufficient": <true or false>,
  "reason": "<one sentence explaining the score>",
  "reformulated_query": "<improved query if score < 3, otherwise null>"
}

Scoring:
5 — Passages directly and completely answer the query
4 — Passages mostly answer with minor gaps  
3 — Passages partially answer (borderline — proceed with caveats)
2 — Passages loosely related but don't answer
1 — Passages irrelevant to the query

Set "sufficient": true if score >= 3.
"""

def evaluate_chunks(query: str, chunks: list[dict]) -> dict:
    """Αξιολογεί αν τα chunks απαντούν το query. Επιστρέφει score, sufficient, reformulated_query."""
    passages = "\n\n".join(
        f"[{i}] {c['title'][:50]}...\n{c['text'][:300]}"
        for i, c in enumerate(chunks, 1)
    )
    user_msg = f"Query: {query}\n\nRetrieved passages:\n{passages}"
    raw = call_llm(EVALUATOR_PROMPT, user_msg)
    print(f"[Evaluator] raw: {raw[:200]}...")

    try:
        return extract_json(raw)
    except (ValueError, json.JSONDecodeError):
        # Fallback: ας πούμε ότι είναι sufficient για να μην κολλήσουμε
        return {"score": 3, "sufficient": True, "reason": "Parse error", "reformulated_query": None}

# Δοκιμή
eval_result = evaluate_chunks("Transformers architecture", test_chunks)
print(f"\nScore:        {eval_result['score']}/5")
print(f"Sufficient:   {eval_result['sufficient']}")
print(f"Reason:       {eval_result['reason']}")
print(f"Reformulated: {eval_result.get('reformulated_query')}")

[Evaluator] raw: {
  "score": 4,
  "sufficient": false,
  "reason": "Passages mostly answer the query, but there are minor gaps in coverage.",
  "reformulated_query": "Transformers architecture with memory-like comput...

Score:        4/5
Sufficient:   False
Reason:       Passages mostly answer the query, but there are minor gaps in coverage.
Reformulated: Transformers architecture with memory-like computation


## Βήμα 3: Το ReAct Loop σε Python

Πρώτα το γράφουμε ως **απλή Python συνάρτηση** (χωρίς LangGraph) ώστε να είναι ξεκάθαρη η λογική.
Μετά θα το μετατρέψουμε σε LangGraph node.

In [7]:
MAX_RETRIES = 5
SCORE_THRESHOLD = 3  # score >= 3: αρκετά καλό

def retriever_evaluator_loop(query: str) -> dict:
    """
    Εκτελεί το ReAct loop:
      1. Retrieve chunks
      2. Evaluate quality
      3. If good enough  return
         If not  reformulate query  retry (max MAX_RETRIES times)
    
    Returns: {"chunks": [...], "retries": int, "final_query": str}
    """
    current_query = query
    
    for attempt in range(1, MAX_RETRIES + 1):
        print(f"\n{'─'*50}")
        print(f"[Attempt {attempt}/{MAX_RETRIES}] Query: '{current_query}'")
        
        chunks = retrieve(current_query)
        
        evaluation = evaluate_chunks(current_query, chunks)
        print(f"[Score] {evaluation['score']}/5 — {evaluation['reason']}")
        
        if evaluation["sufficient"]:
            print(f"[Decision] ✓ Sufficient — proceeding with {len(chunks)} chunks")
            return {"chunks": chunks, "retries": attempt - 1, "final_query": current_query}
        
        if attempt < MAX_RETRIES:
            # Reformulate and retry
            new_query = evaluation.get("reformulated_query") or current_query
            if new_query == current_query:
                # Αν το μοντέλο δεν πρότεινε νέο query, εμείς το τροποποιούμε
                new_query = current_query + " overview methods techniques"
            print(f"[Decision] ✗ Reformulating: '{current_query}'  '{new_query}'")
            current_query = new_query
        else:
            print(f"[Decision] Max retries reached — using best available chunks")
    
    return {"chunks": chunks, "retries": MAX_RETRIES, "final_query": current_query}


# Δοκιμή 1: ερώτηση που θα βρει καλά chunks
print("\n" + "=" * 60)
print("TEST 1: Good query")
result1 = retriever_evaluator_loop("What is attention mechanism in transformers?")
print(f"\nResult: {len(result1['chunks'])} chunks after {result1['retries']} retries")


TEST 1: Good query

──────────────────────────────────────────────────
[Attempt 1/5] Query: 'What is attention mechanism in transformers?'
[Evaluator] raw: {
  "score": 4,
  "sufficient": false,
  "reason": "Passages provide some information about attention mechanism in transformers, but there are minor gaps and unrelated topics.",
  "reformulated_query"...
[Score] 4/5 — Passages provide some information about attention mechanism in transformers, but there are minor gaps and unrelated topics.
[Decision] ✗ Reformulating: 'What is attention mechanism in transformers?'  'What is attention mechanism in transformers? (no improvement needed)'

──────────────────────────────────────────────────
[Attempt 2/5] Query: 'What is attention mechanism in transformers? (no improvement needed)'
[Evaluator] raw: {
  "score": 4,
  "sufficient": true,
  "reason": "Passages provide information about attention mechanisms in transformers, although some details are minor or tangential.",
  "reformulated_quer

In [ ]:
# Δοκιμή 2: ερώτηση που πιθανώς θα πυροδοτήσει reformulation
print("=" * 60)
print("TEST 2: Query that may trigger retry loop")
result2 = retriever_evaluator_loop("quantum computing error correction")
print(f"\nResult: {len(result2['chunks'])} chunks after {result2['retries']} retries")
print(f"Final query: '{result2['final_query']}'")

## Μετατροπή σε LangGraph Node

Τώρα υλοποιούμε την ίδια λογική ως LangGraph node.

Το **κλειδί**: το retry loop δεν γίνεται εσωτερικά με `for` loop —
γίνεται μέσω **conditional edge** που επιστρέφει τον κόμβο στον εαυτό του.
Αυτό επιτρέπει στο LangGraph να παρακολουθεί κάθε iteration ξεχωριστά.

In [9]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class RetrieverState(TypedDict):
    query:         str
    current_query: str        # το τρέχον (πιθανώς reformulated) query
    chunks:        list[dict] # retrieved chunks
    retry_count:   int
    score:         int
    sufficient:    bool
    final_chunks:  list[dict] # validated chunks για τον Synthesizer

def retriever_node(state: RetrieverState) -> dict:
    """ACT: κάνει vector search."""
    q      = state["current_query"] or state["query"]
    chunks = retrieve(q)
    print(f"[Retriever] attempt {state['retry_count']+1}: '{q}'  {len(chunks)} chunks")
    return {"chunks": chunks, "current_query": q}

def evaluator_node(state: RetrieverState) -> dict:
    """OBSERVE + REASON: αξιολογεί τα chunks."""
    evaluation = evaluate_chunks(state["current_query"], state["chunks"])
    score       = evaluation["score"]
    sufficient  = evaluation["sufficient"]
    new_query   = evaluation.get("reformulated_query") or state["current_query"]
    print(f"[Evaluator] score={score}/5, sufficient={sufficient}")
    return {
        "score":         score,
        "sufficient":    sufficient,
        "current_query": new_query,      # αν reformulated, το νέο query για retry
        "retry_count":   state["retry_count"] + 1,
    }

def collect_chunks(state: RetrieverState) -> dict:
    """Φινάλε: αποθηκεύει τα chunks για τον επόμενο agent."""
    print(f"[Collect] Passing {len(state['chunks'])} chunks to Synthesizer")
    return {"final_chunks": state["chunks"]}

def route_after_evaluation(state: RetrieverState) -> str:
    """DECIDE: retry ή proceed;"""
    if state["sufficient"]:
        return "collect"          # αρκετά καλά
    elif state["retry_count"] < MAX_RETRIES:
        return "retry"            # ξαναπροσπαθούμε
    else:
        return "collect"          # εξαντλήσαμε — προχωράμε ό,τι κι αν έχουμε

g = StateGraph(RetrieverState)
g.add_node("retriever",     retriever_node)
g.add_node("evaluator",     evaluator_node)
g.add_node("collect_chunks", collect_chunks)

g.set_entry_point("retriever")
g.add_edge("retriever", "evaluator")

g.add_conditional_edges(
    "evaluator",
    route_after_evaluation,
    {
        "retry":   "retriever",       #  LOOP BACK
        "collect": "collect_chunks",
    }
)
g.add_edge("collect_chunks", END)

retriever_app = g.compile()

# Δοκιμή
INIT = {
    "query": "What is the energy efficiency in Transformers?",
    "current_query": "spiking neural network energy efficiency",
    "chunks": [], "retry_count": 0, "score": 0, "sufficient": False, "final_chunks": []
}
result = retriever_app.invoke(INIT)
print(f"\n✓ Done: {len(result['final_chunks'])} final chunks, {result['retry_count']} evaluation(s)")

[Retriever] attempt 1: 'spiking neural network energy efficiency'  5 chunks
[Evaluator] raw: {
  "score": 4,
  "sufficient": false,
  "reason": "Passages provide relevant information on spiking neural network energy efficiency, but some details are missing or require additional context.",
  "...
[Evaluator] score=4/5, sufficient=False
[Retriever] attempt 2: 'spiking neural network energy efficiency: what methods can be used to improve the efficiency of spiking neural networks?'  5 chunks
[Evaluator] raw: {
  "score": 5,
  "sufficient": true,
  "reason": "Passages collectively provide a comprehensive overview of methods to improve the efficiency of spiking neural networks, including LayerBoost, Superno...
[Evaluator] score=5/5, sufficient=True
[Collect] Passing 5 chunks to Synthesizer

✓ Done: 5 final chunks, 2 evaluation(s)


## Παρατηρήσεις

1. **Ο evaluator κάνει πραγματική κρίση**: δεν επιστρέφει πάντα "sufficient". Δοκιμάστε queries που είναι εκτός corpus.
2. **Το retry loop περνάει από το LangGraph**: κάθε iteration είναι ξεχωριστό graph step.
3. **`MAX_RETRIES` = safety valve**: αποτρέπει infinite loops.
4. **Το `current_query` ενημερώνεται** μετά από κάθε αποτυχημένη αξιολόγηση.

---

Στο επόμενο notebook (`04_synthesizer_agent.ipynb`) παίρνουμε τα validated chunks
και τα μετατρέπουμε σε συνεκτική απάντηση με citations.